# Task 5: Making Visualizations Interactive

## Importing Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

## Main simulation function

In [ ]:
def func(y0, t, alpha, beta, delta, gamma):
    dxdt = (alpha * y0[0]) - (beta * y0[0] * y0[1])
    dydt = (delta * y0[0] * y0[1]) - (gamma * y0[1])
    return dxdt, dydt

## Running the simulation and exploring the results

In [ ]:
# Prey, Predator
y0 = [10, 10]
alpha, beta = 0.1, 0.02
delta, gamma = 0.02, 0.4
end_time = 100
n_samples = 500
t = np.linspace(0, end_time, n_samples)
y = odeint(func, y0, t, args=(alpha, beta, delta, gamma))

## Visualizing time series results

In [ ]:
def plotly_plot_sim(sim):
    fig = go.Figure(
        data=[go.Scatter(x=[i for i in range(len(sim))], y=sim[:, 0],
                         mode="lines", name='Preys',
                         line=dict(width=2, color="blue")),
              go.Scatter(x=[i for i in range(len(sim))], y=sim[:, 1],
                         mode="lines", name='Predators',
                         line=dict(width=2, color="orange"))],
        layout=go.Layout(
            title_text="Predators vs Preys Analysis", hovermode="closest",
            updatemenus=[
                {
                    "buttons": [
                        {
                            "args": [None, {"frame": {"duration": 10, "redraw": False},
                                            "fromcurrent": True,
                                            "transition": {"duration": 1,
                                                           "easing": "quadratic-in-out"}}],
                            "label": "Play",
                            "method": "animate"
                        },
                        {
                            "args": [[None], {"frame": {"duration": 0, "redraw": False},
                                              "mode": "immediate",
                                              "transition": {"duration": 0}}],
                            "label": "Pause",
                            "method": "animate"
                        }
                    ],
                    "direction": "left",
                    "pad": {"r": 10, "t": 87},
                    "showactive": False,
                    "type": "buttons",
                    "x": 0.14,
                    "xanchor": "right",
                    "y": 1.95,
                    "yanchor": "top"
                }
            ]),

        frames=[go.Frame(
            data=[go.Scatter(
                x=[i for i in range(k)],
                y=sim[:, 0],
                mode="lines",
                line=dict(width=2, color="blue")),
                go.Scatter(
                x=[i for i in range(k)],
                y=sim[:, 1],
                mode="lines",
                line=dict(width=2, color="orange"))])

                for k in range(len(sim))],

    )
    fig.update_xaxes(title_text="Number of Days")
    fig.update_yaxes(title_text="Population Size")
    fig.show()

    
plotly_plot_sim(y)

## Visualizing Phase Space plot

In [ ]:
def plotly_phase_space_plot(ranges):
    # Direction fields creation process from: https://scipy-cookbook.readthedocs.io/items/LoktaVolterraTutorial.html
    # Creating a grid and computing the direction at each point
    nb_points = 10
    x = np.linspace(0, 55, nb_points)
    y = np.linspace(0, 30, nb_points)
    X1 , Y1  = np.meshgrid(x, y) 
    # Computing growth rate on the grid
    DX1, DY1 = func([X1, Y1], 0, alpha, beta, delta, gamma)
    # Norm of the growth rate 
    M = (np.hypot(DX1, DY1))    
    # Avoiding zero division errors 
    M[ M == 0] = 1.0
    # Normalizing the arrows
    DX1 /= M      
    DY1 /= M

    fig = ff.create_quiver(X1, Y1, DX1, DY1, scale=1, arrow_scale=0.5, name="Field Direction")

    for y0 in ranges:
        y = odeint(func, y0, t, args=(alpha, beta, delta, gamma))
        fig.add_trace(go.Scatter(x=y[:, 0], y=y[:, 1], name="y0= " + str(y0)))

    fig.update_layout(height=600, width=1200, title_text="Phase Space Plot")
    fig.update_xaxes(title_text="Number of Preys")
    fig.update_yaxes(title_text="Number of Predators")
    fig.show()
    
plotly_phase_space_plot([(10, 10), (10, 12), (20, 20), (30, 25)])

## Creating a timestamp slider

In [ ]:
def make_slider(fig, start, sl_range):
    steps = []
    for i in range(start, len(fig.data), 8):
        step = dict(
            method="update",
            label=str(round(sl_range[len(steps)], 3)),
            args=[{"visible": [False] * len(fig.data)}],
        )
        # Making all traces invisible apart from the 2 that are displayed
        # In the specific slider step selected by the user.
        step["args"][0]["visible"][i] = True # Prey Trace
        step["args"][0]["visible"][i+1] = True # Predator Trace
        steps.append(step)
        
    slider = [dict(
        active=0,
        steps=steps
    )]
    
    return slider

## Interactive Parameter Adjustment

In [ ]:
def adjust_sim_parameters(a_range, b_range, d_range, g_range):
    fig = go.Figure()

    # Creating traces for each simulation created for each of the
    # different parameters values
    for a, b, d, g in zip(a_range, b_range, d_range, g_range):
        y_a = odeint(func, y0, t, args=(a, beta, delta, gamma))
        y_b = odeint(func, y0, t, args=(alpha, b, delta, gamma))
        y_d = odeint(func, y0, t, args=(alpha, beta, d, gamma))
        y_g = odeint(func, y0, t, args=(alpha, beta, delta, g))
        fig.add_traces([
            go.Scatter(
                visible=False,
                line=dict(color="blue", width=6),
                name="Preys",
                x=t,
                y=y_a[:, 0]),
            go.Scatter(
                visible=False,
                line=dict(color="orange", width=6),
                name="Predators",
                x=t,
                y=y_a[:, 1]),
            go.Scatter(
                visible=False,
                line=dict(color="blue", width=6),
                name="Preys",
                x=t,
                y=y_b[:, 0]),
            go.Scatter(
                visible=False,
                line=dict(color="orange", width=6),
                name="Predators",
                x=t,
                y=y_b[:, 1]),
            go.Scatter(
                visible=False,
                line=dict(color="blue", width=6),
                name="Preys",
                x=t,
                y=y_d[:, 0]),
            go.Scatter(
                visible=False,
                line=dict(color="orange", width=6),
                name="Predators",
                x=t,
                y=y_d[:, 1]),
            go.Scatter(
                visible=False,
                line=dict(color="blue", width=6),
                name="Preys",
                x=t,
                y=y_g[:, 0]),
            go.Scatter(
                visible=False,
                line=dict(color="orange", width=6),
                name="Predators",
                x=t,
                y=y_g[:, 1])]
        )

    # Creating a thumbnail frame visible when the plot loads
    fig.data[0].visible = True
    fig.data[1].visible = True

    fig.update_layout(
        title_text="Pick & Adjust Parameter of Interest",
        updatemenus=[
            dict(buttons=list([
                dict(label="None",
                     method="relayout",
                     args=["sliders", []]),
                dict(label="Alpha",
                     method="relayout",
                     # Creating a slider for each of the 4 parameters
                     args=["sliders", make_slider(fig, 0, a_range)]),
                dict(label="Beta",
                     method="relayout",
                     args=["sliders", make_slider(fig, 2, b_range)]),
                dict(label="Delta",
                     method="relayout",
                     args=["sliders", make_slider(fig, 4, d_range)]),
                dict(label="Gamma",
                     method="relayout",
                     args=["sliders", make_slider(fig, 6, g_range)])
            ]),
            )
        ],
        showlegend=True
    )

    fig.show()
    

a_range = np.arange(0.1, 2.1, 0.1)
b_range = np.arange(0.1, 1.1, 0.05)
d_range = np.arange(0.1, 1.1, 0.05)
g_range = np.arange(0.1, 1.1, 0.05)
adjust_sim_parameters(a_range, b_range, d_range, g_range)